# 收益管理问题

**类别：** 仿真

来源： [https://www.hexaly.com/templates/revenue-management-problem](https://www.hexaly.com/templates/revenue-management-problem)


## 问题

**在 Revenue Management Problem 中**，一家公司希望在一个分为若干时段的时间范围内，通过销售某种产品来最大化其收入。它必须决定在时间范围开始时购买的产品总量。然后，在每个时段，它必须决定该时段内销售的产品数量。整个时间范围内销售的产品总数不得超过最初购买的数量。产品价格随时间上涨。为了获得更多利润，公司应当为后续客户保留一些产品，而不是在早期全部售出。为了在每个时段做出最明智的决策，它必须考虑后续时段的需求。由于需求是随机的，公司运行大量仿真以获得对给定单位分配下收入的稳健估计。

	

### 学到的建模原则

- 定义 [external function](https://www.hexaly.com/docs/last/mathematicaloperators/externalfunctions.html#) 以在模型中使用仿真代码
- 启用 [surrogate modeling](https://www.hexaly.com/docs/last/mathematicaloperators/externalfunctions.html#surrogate-modeling) 以处理计算昂贵的外部函数
- 为外部函数指定 [evaluation points](https://www.hexaly.com/docs/last/mathematicaloperators/externalfunctions.html#surrogate-modeling) 的界


## 数据

时间范围由 3 个时段组成，产品的初始成本为 $80。

每个时段 t 的需求由方程 Dₜ=μₜXYₜ 定义，其中：

- Yₜ 服从速率参数 λ=1 的指数分布。
- X 服从形状参数 k=1、尺度参数 θ=1 的 Gamma 分布，等价于标准指数分布。
- μₜ 是该时段的平均需求。

每个时段的价格和平均需求见下表：
**时段****1****2****3**价格100300400平均需求502030每个时段的价格和平均需求
为了获得收入的稳健估计，仿真需要使用 [Monte Carlo method](https://en.wikipedia.org/wiki/Monte_Carlo_method) 运行大量次数（1,000,000 次）。因此每次仿真需要数秒钟。由于该评估函数开销极大，我们不能负担大量运行次数，必须谨慎选择每个评估点。


## 模型

Revenue Management Problem 的 Hexaly 模型使用三个 integer 决策变量。第一个对应于时间范围开始时购买的初始数量。第二个决定为第 2 和第 3 时段预留的产品数量，第三个表示为第 3 时段预留的产品数量。为保证可行性，每个变量都被约束为小于等于前一个变量。

目标函数是仿真函数的返回值。我们需要将其作为 [external function](https://www.hexaly.com/docs/last/mathematicaloperators/externalfunctions.html) 以供模型使用。传递给外部函数的参数是问题的三个 integer 决策变量。由于仿真计算开销很大，我们启用 Hexaly 的 [surrogate modeling features](https://www.hexaly.com/docs/last/mathematicaloperators/externalfunctions.html#surrogate-modeling) 以获得最佳性能。

由于外部函数由用户提供，Hexaly 无法计算该模型的任何下界。然而，如果用户已知其外部函数的下界，可以指定给求解器使用。在本例中，我们知道仿真不会返回负值，因此可以将外部函数的下界设为 0。

使用 surrogate modeling 时，其他一些参数也可能有用。我们可以选择最大评估次数，而不是设置时间限制。我们还可以指定一些先前评估的点以对求解器进行预热。在本例中，我们将函数评估次数限制为 30，并向求解器指出点 (100, 50, 30) 会产生平均收入 4740.99。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import math
import random
from pathlib import Path

from optagent import OptModel, solve


class RevenueManagementFunction:

    def __init__(self, seed):
        self.nb_periods = 3
        self.prices = [100, 300, 400]
        self.mean_demands = [50, 20, 30]
        self.purchase_price = 80
        self.evaluated_points = [{
            "point": [100, 50, 30],
            "value": 4740.99
        }]
        self.nb_simulations = int(1e6)
        self.seed = seed

    # External function
    def evaluate(self, argument_values):
        variables = [argument_values.get(i) for i in range(argument_values.count())]
        # Initial quantity purchased
        nb_units_purchased = variables[0]
        # Number of units that should be left for future periods
        nb_units_reserved = variables[1:] + [0]

        # Set seed for reproducibility
        random.seed(self.seed)
        # Create distribution
        X = [gamma_sample() for i in range(self.nb_simulations)]
        Y = [[exponential_sample() for i in range(self.nb_periods)]
             for j in range(self.nb_simulations)]

        # Run simulations
        sum_profit = 0.0
        for i in range(self.nb_simulations):
            remaining_capacity = nb_units_purchased
            for j in range(self.nb_periods):
                # Generate demand for period j
                demand_j = int(self.mean_demands[j] * X[i] * Y[i][j])
                nb_units_sold = min(
                    max(remaining_capacity - nb_units_reserved[j], 0),
                    demand_j)
                remaining_capacity = remaining_capacity - nb_units_sold
                sum_profit += self.prices[j] * nb_units_sold

        # Calculate mean revenue
        mean_profit = sum_profit / self.nb_simulations
        mean_revenue = mean_profit - self.purchase_price * nb_units_purchased

        return mean_revenue


def exponential_sample(rate_param=1.0):
    u = random.random()
    return math.log(1 - u) / (-rate_param)


def gamma_sample(scale_param=1.0):
    return exponential_sample(scale_param)


def main(output_file=None, time_limit=30, evaluation_limit=None):
    model = OptModel()

        # Generate data
    revenue_management = RevenueManagementFunction(1)
    nb_periods = revenue_management.nb_periods
        # Declare decision variables
    variables = [model.int(0, 100, name=f"period_{i}_quantity") for i in range(nb_periods)]

        # Create the function
    func_expr = model.create_double_external_function(revenue_management.evaluate)
    func_call = func_expr(*variables)

        # Declare constraints
    for i in range(1, nb_periods):
        model.constraint(variables[i] <= variables[i - 1], name=f"reservation_{i}")

    # OptAgent has no surrogate-modeling API; retain the black-box external
    # objective and use the wall-clock limit for the overall solve.
    model.maximize(func_call, name="mean_revenue")
    try:
        solution = solve(model, time_limit_s=float(time_limit) if time_limit is not None else None)
    except RuntimeError as error:
        if "evaluation cancelled" not in str(error).lower():
            raise
        print(f"External simulation cancelled by time limit: {error}")
        return None
    if not solution.feasible:
        print(f"No feasible solution found; Status = {solution.feasible}")
        return solution
    print(f"Mean revenue = {func_call.value}; Status = {solution.feasible}")

        # Write the solution in a file
    if output_file is not None:
        lines = [f"obj={func_call.value}", f"b={variables[0].value}"]
        lines.extend(f"r{i + 1}={variables[i].value}" for i in range(1, nb_periods))
        Path(output_file).write_text("\n".join(lines) + "\n", encoding="utf-8")
    return solution


## 实例调用

该示例不依赖外部实例文件，直接运行仿真收益模型。

In [ ]:
solution_revenue = main(time_limit=1)
getattr(solution_revenue, "feasible", None)
